# Phenotype Observations

Binarizes gene expression (`scBoolSeq`) and aggregates it per phenotype (`AGGREGATION`: `sum`/`mode`/`bootstrap`), producing `meta` (phenotype x gene, values in {0, 1, NaN}).

This step is **PKN-agnostic** — the same `meta` feeds into any PKN-construction notebook (CollecTRI-based **"5. GRN Construction.ipynb"** or data-driven **"5. GRN Construction (FlowGRN).ipynb"**), since which genes are observed to be on/off per phenotype doesn't depend on which regulatory network structure is used downstream.

Exports `meta_{FILE_PREFIX}_bin_aggregated.csv` and `{FILE_PREFIX}_config.json` (`INCLUDE_HC_CONSTRAINT`), consumed by the GRN-construction notebooks.

In [ ]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))
import warnings
warnings.filterwarnings("ignore") # umap deprecation warnings related to numba do not concern us.


In [ ]:
import json

import numpy as np
import pandas as pd

from scboolseq import scBoolSeq

from markers import (
    states_and_markers, # Dictionnary containing phenotypes and their marker genes
    marker_genes, # pandas.Series containing simply the name of markers
)


## Method selection

This notebook consolidates the three aggregation variants (`Sum`, `Mode`, `Bootstrap`).
Change **`AGGREGATION`** below; the rest of the pipeline reuses the same steps with method-specific defaults and file prefixes.

| `AGGREGATION` | Observation tags (default) | Aggregation | Extra constraint |
|---|---|---|---|
| `"sum"` | `pseudotime_inferred_observations.csv` | sum + `SymmetricBinarizer` | — |
| `"mode"` | `pseudotime_inferred_observations.csv` | pandas `.mode()` | — |
| `"bootstrap"` | `scboolseq_inferred_observations.csv` | bootstrap mean + CI | `HC` |

(The PKN source choice that used to live here moved to the GRN-construction notebooks — it's independent of this step.)

In [ ]:
# Choose one of: "sum", "mode", "bootstrap"
AGGREGATION = "sum"

assert AGGREGATION in {"sum", "mode", "bootstrap"}, AGGREGATION

_DEFAULTS = {
    "sum": {
        "obs_tags_file": "pseudotime_inferred_observations.csv",
        "include_hc_constraint": False,
    },
    "mode": {
        "obs_tags_file": "pseudotime_inferred_observations.csv",
        "include_hc_constraint": False,
    },
    "bootstrap": {
        "obs_tags_file": "scboolseq_inferred_observations.csv",
        "include_hc_constraint": True,
    },
}

OBS_TAGS_FILE = _DEFAULTS[AGGREGATION]["obs_tags_file"]
INCLUDE_HC_CONSTRAINT = _DEFAULTS[AGGREGATION]["include_hc_constraint"]
FILE_PREFIX = AGGREGATION  # used for output filenames

print(f"AGGREGATION={AGGREGATION!r}")
print(f"OBS_TAGS_FILE={OBS_TAGS_FILE!r}")
print(f"INCLUDE_HC_CONSTRAINT={INCLUDE_HC_CONSTRAINT}")
print(f"FILE_PREFIX={FILE_PREFIX!r}")


In [ ]:
%ls *csv


In [ ]:
data = pd.read_csv("GSE122466_Retina_vargenes_batch1.csv", index_col=0)
print(data.shape)
data.head()


In [ ]:
obs_tags = pd.read_csv(OBS_TAGS_FILE, index_col=0)
obs_tags.observation.value_counts()


In [ ]:
%%time
scbool = scBoolSeq(
    dor_threshold=0.995, # To retain one of the markers
    confidence=.75 # To maximize binarization
)
scbool.fit(data)


In [ ]:
scbool.criteria_.Category.value_counts()


In [ ]:
scbool.criteria_.loc[marker_genes, :].Category.value_counts()


In [ ]:
%time bin_data = scbool.binarize(data)


In [ ]:
partial_bin_configs = bin_data[marker_genes]
partial_bin_configs.fillna('').head()


In [ ]:
binarized_groups = {}
for obs, frame in obs_tags.groupby('observation'):
    binarized_groups.update({
        obs: bin_data.loc[frame.index, :]
    })


In [ ]:
binarized_groups.keys()


### Aggregate groups

Depending on `AGGREGATION`:

**sum** — encode False/\?/True as −1/0/1, sum within each phenotype, then apply `SymmetricBinarizer`:

$
\begin{equation}
    f(x) =
    \begin{cases}
        -1 &       \text{if}\;\; x = False\\
         0 &       \text{if}\;\;  x = \text{?}\\
         1 &       \text{if}\;\; x = True
    \end{cases}
\end{equation}
$

**mode** — take the first pandas `.mode()` per gene within each phenotype.

**bootstrap** — same −1/0/1 encoding, then bootstrap estimation of the mean; a gene is 1 / 0 / NaN if its CI is entirely positive / entirely negative / otherwise.

In [ ]:
%%time

if AGGREGATION == "sum":
    from scboolseq.binarization import SymmetricBinarizer

    _sym_thresh = .3
    agg_obs = []
    for group_name, binarized_group in binarized_groups.items():
        _mapped_meta = binarized_group.replace(0.0, -1.0).fillna(0.0)
        _meta_bin = (
            SymmetricBinarizer(threshold=_sym_thresh*_mapped_meta.shape[0], require_df=True)
                .fit_transform(
                    _mapped_meta.sum().to_frame()
                )
        )
        _meta_bin.columns = [group_name]
        agg_obs.append(_meta_bin)
    meta = pd.concat(agg_obs, axis=1).T

elif AGGREGATION == "mode":
    agg_obs = []
    for group_name, binarized_group in binarized_groups.items():
        _meta_bin = binarized_group.mode(dropna=False).loc[0, ].to_frame() # Only take the first mode, otherwise there can be multiple rows
        _meta_bin.columns = [group_name]
        agg_obs.append(_meta_bin)
    meta = pd.concat(agg_obs, axis=1).T

elif AGGREGATION == "bootstrap":
    from sklearn.utils import Bunch
    from scboolseq.meta.bootstrap import bootstrap

    def aggregate_bootstrap_frame(b_ci_frame: pd.DataFrame, name: str):
        positive_mask = (b_ci_frame > 0).all(axis=0)
        negative_mask = (b_ci_frame < 0).all(axis=0)
        aggregated = pd.Series(np.nan, index=b_ci_frame.columns, name=name)
        aggregated[positive_mask] = 1.0
        aggregated[negative_mask] = 0.0
        return aggregated

    mapped_binarized_groups = Bunch(**{
        group_name: binarized_group.replace(0.0, -1.0).fillna(0.0)
        for group_name, binarized_group
        in binarized_groups.items()
    })

    bootstrap_frames = {}
    meta_bin_ls = []
    for name, frame in mapped_binarized_groups.items():
        print(name)
        _b = frame.apply(bootstrap)
        bootstrap_frames.update({name: _b})
        meta_bin_ls.append(aggregate_bootstrap_frame(_b, name))

    meta = pd.concat(meta_bin_ls, axis=1).T

else:
    raise ValueError(AGGREGATION)

# Aliases kept for readability / compatibility with the original notebooks
meta_sum = meta_mode = meta_boot = meta_bootstrap_bin = meta
print(AGGREGATION, meta.shape)
meta


In [ ]:
for pheno, genes in states_and_markers.items():
    if pheno in meta.index:
        print(pheno, "\t:\t", meta.loc[pheno, list(genes)].to_dict())


In [ ]:
meta[marker_genes].T


### Export

In [ ]:
_agg_csv = f"meta_{FILE_PREFIX}_bin_aggregated.csv"
meta.to_csv(_agg_csv)
print("Wrote", _agg_csv)


In [ ]:
null_var_mask = meta.var(skipna=False) == 0
null_var_genes = meta.columns[null_var_mask]
null_var_genes.shape


In [ ]:
with open(f"{FILE_PREFIX}_active_invariant_genes.txt", "w") as _active_genes, open(f"{FILE_PREFIX}_inactive_invariant_genes.txt", "w") as _inactive_genes:
    _active_genes.write(f"Gene\n")
    _inactive_genes.write(f"Gene\n")
    for _gene, _val in meta[null_var_genes].mode().T.itertuples(index=True):
        if _val:
            _active_genes.write(f"{_gene}\n")
        else:
            _inactive_genes.write(f"{_gene}\n")


In [ ]:
with open(f"{FILE_PREFIX}_config.json", "w") as _cfg:
    json.dump({"INCLUDE_HC_CONSTRAINT": INCLUDE_HC_CONSTRAINT}, _cfg)
print(f"Wrote {FILE_PREFIX}_config.json")


### Next step

Continue in **"5. GRN Construction.ipynb"** (CollecTRI-based PKN) or **"5. GRN Construction (FlowGRN).ipynb"** (data-driven PKN) — set the same `FILE_PREFIX` there to load `meta_{FILE_PREFIX}_bin_aggregated.csv` and `{FILE_PREFIX}_config.json` exported here.